# 10 · Dateien lesen und schreiben

**Lernziele**

- Du arbeitest mit Pfaden über `pathlib`.
- Du liest und schreibst Textdateien mit `with open(...)`.
- Du verarbeitest CSV-Dateien mit dem Modul `csv`.
- Du speicherst und lädst Daten als JSON.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 7 und 14.

## Pfade mit `pathlib`

`Path` beschreibt einen Datei- oder Ordnerpfad, unabhängig vom Betriebssystem. Mit `/` setzt du Pfade zusammen.

In [ ]:
from pathlib import Path

data_dir = Path("../daten")
shopping_list = data_dir / "einkaufsliste.txt"

print(shopping_list)
print(shopping_list.exists())
print(shopping_list.name, shopping_list.suffix)
print(shopping_list.resolve())  # absoluter Pfad

Ordnerinhalt auflisten:

In [ ]:
for path in sorted(data_dir.iterdir()):
    kind = "[DIR]" if path.is_dir() else f"{path.stat().st_size} Bytes"
    print(f"{path.name:<25} {kind}")

## Textdateien lesen

`with open(...)` öffnet die Datei und schließt sie am Ende des Blocks automatisch, auch bei Fehlern. Gib immer `encoding="utf-8"` an. Seit Python 3.15 ist UTF-8 zwar überall die Voreinstellung, ältere Versionen nehmen unter Windows aber eine andere Kodierung, und dann gehen Umlaute kaputt. *Explicit is better than implicit:* Mit `encoding="utf-8"` läuft dein Code mit jeder Version gleich.

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    content = file.read()
print(content)

Zeile für Zeile lesen ist bei großen Dateien sparsamer:

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line.strip()}")

Für kleine Dateien gibt es eine Abkürzung:

In [ ]:
shopping_list.read_text(encoding="utf-8").splitlines()

## Textdateien schreiben

Der **Modus** bestimmt, was passiert:

| Modus | Bedeutung |
| --- | --- |
| `"r"` | lesen (Standard) |
| `"w"` | schreiben, **überschreibt** vorhandenen Inhalt |
| `"a"` | anhängen |

In [ ]:
output_dir = Path("../daten/ausgabe")
output_dir.mkdir(exist_ok=True)
notes = output_dir / "notizen.txt"

with open(notes, "w", encoding="utf-8") as file:
    file.write("Erste Zeile\n")
    file.write("Zweite Zeile\n")

with open(notes, "a", encoding="utf-8") as file:
    file.write("Angehängte Zeile\n")

print(notes.read_text(encoding="utf-8"))

## CSV-Dateien

CSV ist das Austauschformat für Tabellen, etwa aus Excel. Das Modul `csv` kümmert sich um Trennzeichen und Anführungszeichen. `DictReader` liefert jede Zeile als Dictionary mit den Spaltennamen als Schlüssel.

In [ ]:
import csv

with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        print(f"{row['name']:<10} {row['points']:>5} Punkte, Level {row['level']}")

> Werte aus CSV sind immer Strings. Zum Rechnen umwandeln: `int(row["points"])`.

Schreiben mit `DictWriter`:

In [ ]:
results = [
    {"name": "Ada", "points": 1200},
    {"name": "Linus", "points": 950},
]

with open(output_dir / "ergebnisse.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "points"])
    writer.writeheader()
    writer.writerows(results)

print((output_dir / "ergebnisse.csv").read_text(encoding="utf-8"))

## JSON

JSON speichert verschachtelte Daten aus Listen und Dictionaries. Es ist das Standardformat von Web-APIs.

In [ ]:
import json

with open(data_dir / "personen.json", encoding="utf-8") as file:
    data = json.load(file)

for person in data["people"]:
    print(f"{person['name']} aus {person['city']}")

In [ ]:
data["people"].append({"name": "Grace", "city": "New York"})

with open(output_dir / "personen_neu.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2, ensure_ascii=False)

print((output_dir / "personen_neu.json").read_text(encoding="utf-8"))

## Übungen

**Übung 1 (Zeilen zählen):** Zähle, wie viele Zeilen und Wörter `../daten/faust.txt` enthält.

In [ ]:
# Deine Lösung


**Übung 2 (Highscores auswerten):** Lies `../daten/highscores.csv` und gib den Spieler mit den meisten Punkten sowie den Punkteschnitt aus.

In [ ]:
# Deine Lösung


**Übung 3 (Einkaufsliste):** Schreibe eine Funktion `add_item(path, item)`, die einen Eintrag an eine Einkaufsliste anhängt, und eine Funktion `read_items(path)`, die alle Einträge als Liste zurückgibt. Nutze eine eigene Datei in `../daten/ausgabe/`.

In [ ]:
# Deine Lösung


**Übung 4 (Wortstatistik als JSON):** Zähle die fünf häufigsten Wörter in `faust.txt` (klein geschrieben, ohne Satzzeichen) und speichere das Ergebnis als JSON in `../daten/ausgabe/woerter.json`.

In [ ]:
# Deine Lösung


## Zusammenfassung

- `Path("ordner") / "datei.txt"` baut Pfade, `.exists()`, `.iterdir()`, `.mkdir()` helfen beim Umgang.
- Immer `with open(pfad, modus, encoding="utf-8") as file:`.
- Modus `"r"` lesen, `"w"` überschreiben, `"a"` anhängen.
- CSV: `csv.DictReader` und `csv.DictWriter`, dabei `newline=""` angeben.
- JSON: `json.load` / `json.dump(..., indent=2, ensure_ascii=False)`.